# Build Your Own Mini Laya From Scratch

**What you'll learn**
- Build Laya's architecture on a plain ModernBERT encoder: your own Mini Laya
- Turn Kev's labelled eval sets into typed training questions
- Laya's loss: a reward from a strictly proper scoring rule
- Train it with the same `finetune` as the previous section
- Compare your Mini Laya with Laya
- Lab: fine-tune your Mini Laya on the card problem

## Setup

In [ ]:
import os, pathlib
import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoTokenizer
import laya
from laya.common import build_model, proper_reward, QTYPES
from laya.train import save_checkpoint, finetune, TrainConfig
from laya.evals import Dataset, evaluate

ENC = "answerdotai/ModernBERT-base"
EV = "../vendor/kev/evals"

## The Mini Laya Architecture and Its Config
- The config describes the model: which encoder, how many head layers, the input lengths
- `build_model`: the pretrained encoder plus a new, random decision head
- `save_checkpoint`: an untrained Laya checkpoint, the starting point for `finetune`
- Fix the seed: some random heads sit on a training plateau far longer than others

In [ ]:
CONFIG = {
    "encoder": ENC,  # the pretrained encoder that reads the text
    "head_layers": 2,  # layers in the new decision head
    "max_len": 512,  # the whole input, in tokens
    "head_max_len": 192,  # tokens kept for the question and its options
    "act_costs": {"escalate": 0.5},  # a second action, escalate, and what it costs
    "amp_dtype": "bf16",  # the 16-bit format laya.Agent runs in on a GPU
    "model_name": "mini-laya",  # the name the model reports in its answers
}
CONFIG

In [ ]:
torch.manual_seed(0)  # the same random head on every run
model = build_model(CONFIG)  # the pretrained encoder plus a new, random decision head
model.head

In [ ]:
tok = AutoTokenizer.from_pretrained(ENC)
save_checkpoint(model, tok, CONFIG, "runs/mini-laya-untrained")  # the starting point for finetune
os.listdir("runs/mini-laya-untrained")

In [ ]:
total = 0
for p in model.parameters():
    total += p.numel()  # every weight in the model

encoder = model.encoder.num_parameters()  # the pretrained part
head = total - encoder  # the new part
encoder, head, total

## Build the Training Data From Kev's Eval Sets
- Every Kev record: a `state` and typed questions, each with its gold `label`
- One row per question: a record that asks two or three questions becomes two or three rows
- The label can stay in the question, Laya ignores it; a copy goes to `expected`, the task name to `tags`

In [ ]:
files = sorted(pathlib.Path(EV).rglob("*.jsonl"))  # every Kev eval file, in all subfolders
len(files)

In [ ]:
records = pd.read_json(f"{EV}/v4/transfer-v4/development.jsonl", lines=True)  # one Kev file as a table
records.head()

In [ ]:
record = records.iloc[0]  # one Kev record
record["questions"]

In [ ]:
rows = []
for path in files:
    records = pd.read_json(path, lines=True)  # one Kev file as a table
    for record in records.to_dict("records"):  # one Kev record at a time
        for qid, question in record["questions"].items():  # a record can ask several questions
            rows.append({
                "state": record["state"],
                "questions": {qid: question},  # one question per row
                "expected": {qid: question["label"]},  # the gold answer, where Laya looks for it
                "tags": [question["src"]],  # the task name, so evaluate reports each task
                "qid": qid,
                "task": question["src"],
                "type": question["type"],
                "text": str(record["state"]),  # the state as text, to spot held-out rows
                "file": path.as_posix(),  # the Kev file this row came from
            })
len(rows)

In [ ]:
kev = pd.DataFrame(rows)
kev.head()

- Hold out: the pool calibration and test sets, the transfer set, Banking77 test messages
- Training rows: every other question whose text is not held out and whose task is not a transfer task, each once

In [ ]:
HELD = {
    "calibration": f"{EV}/public-pool-v6/calibration.jsonl",
    "test": f"{EV}/public-pool-v6/test.jsonl",
    "transfer": f"{EV}/v4/transfer-v4/development.jsonl",
}

splits = {}
for name, path in HELD.items():
    splits[name] = kev[kev["file"] == path]  # the held-out sets, straight from their files
    print(name, len(splits[name]))

In [ ]:
held = pd.concat(splits.values())  # the three held-out sets in one table
held_texts = held["text"]  # texts that must never be trained on
len(held_texts)

In [ ]:
card_test = load_dataset("mteb/banking77", split="test")  # the card test messages, kept for the Lab
card_texts = card_test["text"]
len(card_texts)

In [ ]:
transfer_tasks = splits["transfer"]["task"].unique()  # tasks that must stay unseen
transfer_tasks

In [ ]:
is_held = kev["text"].isin(held_texts)
is_card_test = kev["text"].isin(card_texts)
is_transfer_task = kev["task"].isin(transfer_tasks)
is_held.sum(), is_card_test.sum(), is_transfer_task.sum()

In [ ]:
train = kev[~is_held & ~is_card_test & ~is_transfer_task]  # keep a row only if none of the three match
len(train)

In [ ]:
train = train.drop_duplicates(["text", "qid"])  # the same question on the same text only once
splits["train"] = train
len(train)

In [ ]:
train["type"].value_counts()

In [ ]:
train["task"].nunique()

In [ ]:
train.iloc[0]

- Save every split as JSONL: `finetune` and `evaluate` read it, and so does the Mini Clef section

In [ ]:
os.makedirs("runs/system-one-data", exist_ok=True)
columns = ["state", "questions", "expected", "tags"]  # what finetune and evaluate read

for name, split in splits.items():
    path = f"runs/system-one-data/{name}.jsonl"
    split[columns].to_json(path, orient="records", lines=True)
    print(path, len(split))

## Baseline: The Untrained Model
- The untrained checkpoint loads like any Laya
- The head is random: accuracy near chance

In [ ]:
test = Dataset.from_jsonl("runs/system-one-data/test.jsonl")
len(test)

In [ ]:
untrained = laya.Agent("runs/mini-laya-untrained", device="cuda")
untrained.cfg["model_name"]

In [ ]:
report = evaluate(untrained, test)  # every test row through the untrained model
report.overall.keys()

In [ ]:
metrics = ["choice_accuracy", "noul_accuracy", "score_mae", "ece"]
overall = pd.Series(report.overall)
overall[metrics]

## Laya's Loss: Proper Scoring Rules Reward Honest Confidence
- Reward = log score + spherical score, minus ranked probability score for `score` questions
- `finetune`'s default loss, `rlcd`, pushes the model toward answers with a higher reward
- A proper scoring rule pays most for reporting what you actually believe

In [ ]:
target = torch.tensor([[0., 1., 0., 0.]])  # the right answer is the second option
qtype = torch.tensor([QTYPES["choice"]])  # a choice question
mask = torch.ones(1, 4)  # all four options are real
target, qtype, mask

In [ ]:
REPORTS = {
    "confident, right": [0.01, 0.97, 0.01, 0.01],
    "hedged, right": [0.1, 0.6, 0.2, 0.1],
    "uniform": [0.25, 0.25, 0.25, 0.25],
    "confident, wrong": [0.97, 0.01, 0.01, 0.01],
}

In [ ]:
for name, probs in REPORTS.items():
    reward = proper_reward(torch.tensor([probs]), target, qtype, mask)  # how much this answer earns
    print(name, round(reward.item(), 3))

In [ ]:
def reward(q, y):
    probs = torch.tensor([[1 - q, q]])  # the model says "yes" with probability q
    truth = torch.tensor([[1 - y, y]])  # the real answer: y = 1 for yes, 0 for no
    noul = torch.tensor([QTYPES["noul"]])  # a yes or no question
    return proper_reward(probs, truth, noul, torch.ones(1, 2)).item()

reward(0.9, 1), reward(0.9, 0)

In [ ]:
belief = 0.7  # the model believes the answer is "yes" 70% of the time

for q in [0.5, 0.6, 0.7, 0.8, 0.9]:
    average = belief * reward(q, 1) + (1 - belief) * reward(q, 0)  # its average reward if it says q
    print(q, round(average, 4))

## Train Mini Laya With Laya's finetune
- The same call as fine-tuning Laya, now from the untrained checkpoint
- `TrainConfig` defaults: two learning rates, gentle for the encoder and faster for the new head, `rlcd` loss
- `shuffle_options`: choice options in a new order every epoch, so the model cannot learn positions
- Temperatures: fitted per question type on a hold-out of the training rows (10%, at most 400)

In [ ]:
config = TrainConfig(shuffle_options=("choice",), eval_data="runs/system-one-data/test.jsonl")
config

In [ ]:
OUT = "runs/mini-laya"
run = finetune("runs/system-one-data/train.jsonl", "runs/mini-laya-untrained", OUT, config)
run.keys()

In [ ]:
run["train_items"], run["calibration_items"], run["skipped"]

In [ ]:
run["epoch_loss"]

In [ ]:
run["temperature"]

In [ ]:
before = run["train_report"]["before"]  # the test scores before training
after = run["train_report"]["after"]  # and after
table = pd.DataFrame([before, after], index=["before", "after"])
table = table.drop(columns="by_type")
table

In [ ]:
pd.DataFrame.from_dict(after["by_type"], orient="index")

In [ ]:
ours = laya.Agent(OUT, device="cuda")
ours.cfg["model_name"]

In [ ]:
QUESTIONS = {
    "sentiment": {
        "type": "score",
        "instructions": "How does the customer feel?",
        "criteria": ["very negative", "negative", "neutral", "positive", "very positive"],
    },
    "topic": {
        "type": "choice",
        "instructions": "What is it about?",
        "criteria": ["travel", "banking", "food"],
    },
    "refund": {
        "type": "noul",
        "instructions": "Is the customer likely to ask for a refund?",
    },
}

In [ ]:
result = ours.system_one("My flight was cancelled and nobody at the desk would help.", QUESTIONS)
result["answers"]

## Mini Laya vs Laya: Accuracy on the Same Test Sets
- `evaluate`, the same test and transfer rows, each model with its own temperatures

In [ ]:
transfer = Dataset.from_jsonl("runs/system-one-data/transfer.jsonl")
original = laya.Agent("convaiinnovations/laya", device="cuda")
len(transfer)

In [ ]:
AGENTS = {
    "Mini Laya (ModernBERT-base)": ours,
    "Laya (ModernBERT-large)": original,
}
SETS = {
    "test": test,
    "transfer": transfer,
}

In [ ]:
results = []
for model_name, agent in AGENTS.items():
    for set_name, data in SETS.items():
        report = evaluate(agent, data)  # every row through this model
        scores = pd.Series(report.overall, name=f"{model_name}, {set_name}")
        results.append(scores[metrics])  # one row of the table
        print(model_name, set_name, "done")

In [ ]:
pd.DataFrame(results)

## Lab: Fine-Tune Mini Laya on Custom Data
- The card rows saved by the previous section
- Kev's sets include Banking77 training messages (test messages were held out): the first score is not a true zero-shot
- `finetune` again, now from your Mini Laya

In [ ]:
lab_config = TrainConfig(eval_data="runs/cards/test.jsonl")
cards = finetune("runs/cards/train.jsonl", OUT, "runs/mini-laya-cards", lab_config)
cards.keys()

In [ ]:
before = cards["train_report"]["before"]
after = cards["train_report"]["after"]
table = pd.DataFrame([before, after], index=["Mini Laya", "Mini Laya + cards"])
table = table.drop(columns="by_type")
table

## Cheat Sheet
| Step | Code |
|---|---|
| Architecture | `build_model({"encoder": ENC, "head_layers": 2, ...})` |
| Starting point | `save_checkpoint(model, tok, config, path)` |
| Data | Kev rows to `state`, `questions`, `expected`, `tags`; held-out texts removed |
| Loss | `rlcd`: reward from `proper_reward`, a strictly proper scoring rule |
| Train | `finetune(data, untrained, out, TrainConfig(shuffle_options=("choice",)))` |
| Calibration | inside `finetune`: one temperature per question type |
| Compare | `evaluate(agent, Dataset.from_jsonl(path))` |
| Load | `laya.Agent("runs/mini-laya")` |